In [ ]:
import numpy as np
import sounddevice as sd
import ipywidgets as widgets
from IPython.display import display
import threading

fs = 44100
buffer_size = 1024

class RealTimeAudioPlayer:
    def __init__(self, sample_rate=44100, buffer_size=1024):
        self.sample_rate = sample_rate
        self.buffer_size = buffer_size
        self.is_playing = False
        self.stream = None
        self.phase = 0.0
        
        # Audio parameters
        self.frequency = 440.0
        self.amplitude = 0.5
        self.phase_offset = 0.0  # radians
        
        # Lock for thread safety
        self.lock = threading.Lock()
    
    def audio_callback(self, outdata, frames, time_info, status):
        """
        Real-time audio callback function
        
        Phase Calculation Process:
        1. Basic sine wave: y(t) = A * sin(2π * f * t + φ)
        2. For discrete samples: y[n] = A * sin(2π * f * n/fs + φ)
        3. Phase increment per sample: Δφ = 2π * f / fs
        4. Phase for sample n: φ[n] = φ[0] + n * Δφ
        5. Continuous phase: φ[n+1] = φ[n] + Δφ (mod 2π)
        
        Where:
        - A = amplitude
        - f = frequency (Hz)
        - t = time (seconds)
        - n = sample index
        - fs = sample rate
        - φ = phase (radians)
        - Δφ = phase increment
        """
        if status:
            print(f"Stream status: {status}")
        
        with self.lock:
            # Calculate phase increment per sample
            # Δφ = 2π * f / fs
            phase_increment = 2 * np.pi * self.frequency / self.sample_rate
            
            # Generate phase array for current buffer
            # φ[n] = current_phase + n * Δφ
            sample_indices = np.arange(frames)
            phase_array = self.phase + sample_indices * phase_increment
            
            # Generate waveform with phase offset
            # y[n] = A * sin(φ[n] + φ_offset)
            wave = self.amplitude * np.sin(phase_array + self.phase_offset)
            
            # Update phase for next callback (continuous phase)
            # φ_next = φ_last + Δφ (mod 2π)
            self.phase = phase_array[-1] + phase_increment
            self.phase = self.phase % (2 * np.pi)  # Keep phase in [0, 2π]
        
        # Convert to float32 and copy to output buffer
        outdata[:, 0] = wave.astype(np.float32)
    
    def start(self):
        """Start real-time audio playback"""
        if not self.is_playing:
            self.is_playing = True
            self.stream = sd.OutputStream(
                samplerate=self.sample_rate,
                channels=1,
                blocksize=self.buffer_size,
                callback=self.audio_callback
            )
            self.stream.start()
            print("Audio playback started")
    
    def stop(self):
        """Stop audio playback"""
        if self.is_playing:
            self.is_playing = False
            if self.stream:
                self.stream.stop()
                self.stream.close()
                self.stream = None
            print("Audio playback stopped")
    
    def set_frequency(self, freq):
        """Set frequency in Hz"""
        with self.lock:
            self.frequency = float(freq)
    
    def set_amplitude(self, amp):
        """Set amplitude (0.0 to 1.0)"""
        with self.lock:
            self.amplitude = np.clip(float(amp), 0.0, 1.0)
    
    def set_phase_offset(self, phase):
        """Set phase offset in radians"""
        with self.lock:
            self.phase_offset = float(phase)
    
    def get_frequency(self):
        """Get current frequency"""
        with self.lock:
            return self.frequency
    
    def get_amplitude(self):
        """Get current amplitude"""
        with self.lock:
            return self.amplitude
    
    def get_phase_offset(self):
        """Get current phase offset"""
        with self.lock:
            return self.phase_offset

# Create audio player instance
player = RealTimeAudioPlayer(fs, buffer_size)

# Create UI widgets
freq_slider = widgets.FloatSlider(
    value=440,
    min=20,
    max=2000,
    step=1,
    description="Frequency (Hz)",
    continuous_update=True
)

amplitude_slider = widgets.FloatSlider(
    value=0.5,
    min=0,
    max=1,
    step=0.01,
    description="Amplitude",
    continuous_update=True
)

phase_slider = widgets.FloatSlider(
    value=0,
    min=0,
    max=2*np.pi,
    step=0.01,
    description="Phase (rad)",
    continuous_update=True
)

start_button = widgets.Button(description="Start Playback")
stop_button = widgets.Button(description="Stop Playback")

# Event handlers
def on_freq_change(change):
    player.set_frequency(change['new'])

def on_amplitude_change(change):
    player.set_amplitude(change['new'])

def on_phase_change(change):
    player.set_phase_offset(change['new'])

def on_start_click(b):
    player.start()

def on_stop_click(b):
    player.stop()

# Connect handlers
freq_slider.observe(on_freq_change, names='value')
amplitude_slider.observe(on_amplitude_change, names='value')
phase_slider.observe(on_phase_change, names='value')
start_button.on_click(on_start_click)
stop_button.on_click(on_stop_click)

# Display interface
controls = widgets.VBox([
    widgets.Label("Real-Time Audio Synthesizer"),
    freq_slider,
    amplitude_slider,
    phase_slider,
    widgets.HBox([start_button, stop_button])
])

display(controls)

# Phase Calculation Explanation

## Basic Formulas

**Continuous Time Form:**
```
y(t) = A * sin(2π * f * t + φ)
```

**Discrete Time Form:**
```
y[n] = A * sin(2π * f * n/fs + φ)
```

## Phase Increment Calculation

**Phase Increment per Sample:**
```
Δφ = 2π * f / fs
```

**Continuous Phase Update:**
```
φ[n+1] = (φ[n] + Δφ) mod 2π
```

## Parameters

- **A**: Amplitude (0.0 to 1.0)
- **f**: Frequency in Hz
- **t**: Time in seconds
- **n**: Sample index (0, 1, 2, ...)
- **fs**: Sample rate (44100 Hz)
- **φ**: Phase in radians (0 to 2π)
- **Δφ**: Phase increment per sample

## Example Calculation

For 440 Hz at 44100 Hz sample rate:
```
Δφ = 2π * 440 / 44100 ≈ 0.0314 radians per sample
```

## Implementation Steps

1. Calculate phase increment: `Δφ = 2π * f / fs`
2. Generate phase array: `φ[n] = current_phase + n * Δφ`
3. Generate waveform: `y[n] = A * sin(φ[n] + φ_offset)`
4. Update continuous phase: `φ_next = (φ_last + Δφ) mod 2π`

This ensures seamless phase continuity across audio buffers for real-time synthesis.